# Refinement through frozen decoders

The appendix on refinement through frozen decoders (Tables 18 and 19).

The AE, VAE and VQ-VAE store one latent per snapshot and share a pretrained decoder. Their latent can be refined with the objective of GyroSplat's last stages without retraining anything: 400 Adam steps on the field, potential and flux terms (each relative to its starting value, weights 1, ½, ½), with the latent in units of its starting standard deviation; then, as two separate alternatives from that result, the Gauss-Newton refinement of the flux (fixed damping 1e-8 · max diag JJᵀ) or an adaptive Levenberg-Marquardt variant (damping from 1e-4, a step is kept only when the flux residual falls). The AE and VAE payload does not change; the refined VQ-VAE latent is no longer a codebook entry and is stored as fp32 values (294,912 bytes, the same as the AE).

In [1]:
import pathlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = pathlib.Path.cwd()
while not (REPO / "neugk").exists():  # the repository root
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
RESULTS = REPO / "neugk/gyrosplat/results"
CONFIGS = REPO / "configs/gyrosplat"

from neugk.gyrosplat.eval.paper import experiments as X
from neugk.gyrosplat.eval.paper import load_experiments, load_rows

rows = load_rows(RESULTS / "rows/gkw_rows.jsonl")
exp = load_experiments(RESULTS / "experiments")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(exist_ok=True)

## Before and after (Table 18)

Medians over the 39 snapshots at 302×. The Adam stage improves all three models; forcing the flux error to zero with Gauss-Newton or Levenberg-Marquardt costs field and potential accuracy in these fits.

In [2]:
display(Markdown(X.latent_table(exp, rows)))
med = X.latent_medians(exp)
for model in ("autoencoder", "vae", "vqvae"):
    print(f"{model}: Adam {med[(model, 'adam')]['seconds']:.0f} s, Gauss-Newton {med[(model, 'gauss_newton')]['seconds']:.0f} s, "
          f"Levenberg-Marquardt {med[(model, 'levenberg_marquardt')]['seconds']:.0f} s per snapshot (median, one GPU)")

| model | fit | PSNR_f | PSNR_φ | flux relL1 | κ/κ_truth | Q(v∥,μ) |
|---|---|---|---|---|---|---|
| AE | Initial | 35.49 | 15.42 | 0.131 | 0.864 | 0.108 |
| AE | 400 Adam steps | 35.88 | 26.57 | 0.032 | 0.857 | 0.111 |
| AE | + GN | 34.69 | 10.20 | 1.7×10⁻⁷ | 0.889 | 0.111 |
| AE | + LM | 34.91 | 10.33 | 1.5×10⁻⁷ | 0.888 | 0.109 |
| VAE | Initial | 34.35 | 9.33 | 0.193 | 0.819 | 0.134 |
| VAE | 400 Adam steps | 34.75 | 24.18 | 0.063 | 0.816 | 0.140 |
| VAE | + GN | 27.66 | -9.76 | 4.7×10⁻⁴ | 1.355 | 0.307 |
| VAE | + LM | 33.68 | 5.96 | 0.036 | 0.852 | 0.138 |
| VQ-VAE | Initial | 34.16 | 12.91 | 0.218 | 0.793 | 0.205 |
| VQ-VAE | 400 Adam steps | 34.50 | 22.97 | 0.087 | 0.793 | 0.112 |
| VQ-VAE | + GN | 27.19 | -2.82 | 2.8×10⁻⁷ | 0.936 | 0.164 |
| VQ-VAE | + LM | 33.08 | 11.62 | 0.033 | 0.832 | 0.100 |
| GyroSplat | 5-seed mean | 37.63 | 27.69 | 5.2×10⁻⁷ | 0.984 | 0.433 |

autoencoder: Adam 22 s, Gauss-Newton 9 s, Levenberg-Marquardt 8 s per snapshot (median, one GPU)
vae: Adam 22 s, Gauss-Newton 9 s, Levenberg-Marquardt 9 s per snapshot (median, one GPU)
vqvae: Adam 22 s, Gauss-Newton 9 s, Levenberg-Marquardt 9 s per snapshot (median, one GPU)


## Longer refinement (Table 19)

2,000 instead of 400 Adam steps on the nine pilot snapshots that have both runs (medians over those nine). The longer fit improves all three models, so Table 18 compares at a fixed compute budget.

In [3]:
display(Markdown(X.latent_steps_table(exp)))

| model | PSNR_φ (dB) | flux relL1 | PSNR_f (dB) |
|---|---|---|---|
| AE | 25.88 → 26.78 | 0.032 → 0.011 | 35.83 → 36.16 |
| VAE | 22.89 → 24.83 | 0.067 → 0.035 | 34.65 → 35.44 |
| VQ-VAE | 22.13 → 23.34 | 0.083 → 0.051 | 34.41 → 34.95 |